# 03 — Model Training & Evaluation
Train a Random Forest classifier on all 15 features across all 114 genres.
The feature importance output from this run tells us what to cut next iteration.

In [ ]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

%matplotlib inline
sns.set_theme(style="whitegrid")

In [ ]:
# Cell 2 — Load processed arrays and label encoder
X_train = np.load("../data/processed/X_train.npy")
X_test  = np.load("../data/processed/X_test.npy")
y_train = np.load("../data/processed/y_train.npy")
y_test  = np.load("../data/processed/y_test.npy")
le      = joblib.load("../models/label_encoder.pkl")

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Classes : {len(le.classes_)} genres")

In [ ]:
# Cell 3 — Train RandomForestClassifier
# n_estimators=200: 200 trees — good balance of accuracy vs training time
# n_jobs=-1: use all CPU cores to parallelize training
# random_state=42: reproducible results
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print("Training complete.")

In [ ]:
# Cell 4 — Evaluate: classification report (F1, precision, recall per genre)
# F1 is our primary metric because genres are imbalanced in general;
# here they're balanced, but F1 still gives a clearer per-class picture than accuracy
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Cell 5 — Confusion matrix heatmap
# With 114 genres the matrix is 114x114 — too dense for per-cell annotations.
# We use color only: bright = many predictions, dark = few.
# Diagonal = correct predictions; off-diagonal = misclassifications.
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(22, 20))
sns.heatmap(
    cm,
    annot=False,              # no numbers — too many cells
    cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title("Confusion Matrix — Random Forest (114 Genres)", fontsize=14)
plt.xlabel("Predicted Genre")
plt.ylabel("True Genre")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Feature importance bar chart
# This is the key output of this run: which of the 15 features does the
# Random Forest rely on most? Low-importance features are candidates to cut
# in the next iteration.
ALL_FEATURES = [
    'popularity','danceability', 'energy','duration_ms', 'mode',
    'instrumentalness', 'liveness', 'valence', 'tempo', 'loudness',
    'speechiness', 'acousticness'
]

importances = pd.Series(rf.feature_importances_, index=ALL_FEATURES).sort_values()

plt.figure(figsize=(9, 6))
importances.plot(kind="barh", color="steelblue")
plt.title("Feature Importances — Random Forest (All 15 Features)", fontsize=14)
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("\nFeature importances (descending):")
for feat, imp in importances.sort_values(ascending=False).items():
    print(f"  {feat:<20} {imp:.4f}")

In [ ]:
# Cell 7 — Save trained model
joblib.dump(rf, "../models/random_forest.pkl")
print("Model saved to models/random_forest.pkl")